# FlowGuard - Full Evaluation
Comprehensive evaluation across all phases and protocols.

In [ ]:
import os, sys

REPO_DIR = "/content/flowguard"
if not os.path.exists(REPO_DIR):
    raise RuntimeError("Run notebook 00_setup_and_validate.ipynb first.")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import json
from src.utils.config import load_config, get_device
from src.utils.reproducibility import setup_reproducibility
from src.evaluation.protocols import generate_comparison_table
from src.evaluation.cross_dataset import run_full_evaluation

config = load_config("configs/base.yaml")
setup_reproducibility(config)
device = get_device(config)
print(f"Device: {device}")

## Evaluate All Phases

In [ ]:
from src.models.mlp_baseline import MLPBaseline
from src.models.flowguard import FlowGuard
from src.data.features import get_input_dim

all_results = {}

# Phase 1 - MLP Baseline (if available)
for ds_info in config['data']['datasets']:
    name = ds_info['name']
    ckpt = f"checkpoints/phase1/{name}/best_model.pt"
    if os.path.exists(ckpt):
        model = MLPBaseline(input_dim=get_input_dim(), num_classes=2)
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state['model_state_dict'])
        model.to(device)
        results = run_full_evaluation(model, config, phase_name=f"phase1_{name}")
        all_results[f"phase1_{name}"] = results
        break  # Just evaluate first available

# Phase 3 - Federated
ckpt = "checkpoints/phase3/final_global.pt"
if os.path.exists(ckpt):
    model = FlowGuard(config)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.to(device)
    results = run_full_evaluation(model, config, phase_name="phase3_federated")
    all_results["phase3"] = results

# Phase 5 - Hardened
ckpt = "checkpoints/phase5/hardened_model.pt"
if os.path.exists(ckpt):
    model = FlowGuard(config)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.to(device)
    results = run_full_evaluation(model, config, phase_name="phase5_hardened")
    all_results["phase5"] = results

## Comparison Table

In [ ]:
# Flatten results for comparison table
from src.evaluation.protocols import generate_comparison_table, ProtocolResult

flat_results = {}
for phase, phase_results in all_results.items():
    flat_results[phase] = []
    for protocol_key, result_list in phase_results.items():
        if isinstance(result_list, list):
            flat_results[phase].extend(result_list)

table = generate_comparison_table(flat_results)
print(table)

# Save
with open("results/metrics/comparison_table.md", "w") as f:
    f.write(table)

## SHAP Analysis

In [ ]:
from src.evaluation.explainability import compute_shap_values, plot_shap_summary, get_feature_importance
from src.data.features import get_feature_names

# Use best available model
best_model = model  # Last loaded model

test_path = "data/splits/protocol_a/unsw_test.parquet"
if os.path.exists(test_path):
    from src.data.dataset import create_dataloader
    loader = create_dataloader(test_path, batch_size=256, shuffle=False, num_workers=0)
    
    print("Computing SHAP values (this may take a while)...")
    result = compute_shap_values(best_model, loader, get_feature_names(), device, num_samples=200)
    
    if result:
        shap_values, data, names = result
        plot_shap_summary(shap_values, data, names, save_path="results/shap/summary.png")
        
        importance = get_feature_importance(shap_values, names)
        print("\nTop 10 Most Important Features:")
        for feat, imp in importance[:10]:
            print(f"  {feat}: {imp:.4f}")